In [1]:
from csv import DictReader
from functools import partial
from itertools import islice
from csv_reader import CSVBatchReader
from icecream import ic

In [2]:
ic.configureOutput(prefix="", outputFunction=print)

In [6]:
with open("./data/sample.csv", "wb") as f:
    f.write(b"id#!name#!signup_date#!score#!active\n")
    f.write(b"1#!Alice#!2023-01-15#!85.5#!True\n")
    f.write(b"2#!Bob#!2023-03-22#!92.0#!False\n")
    f.write(b"3#!Charlie#!2023-05-10#!78.0#!True\n")
    f.write(b"4#!David#!2023-07-30#!88.5#!False\n")
    f.write(b"5#!Eva#!2023-10-01#!95.0#!True\n")
    # Inject a truly invalid UTF-8 byte: 0x96
    f.write(b"6#!InvalidChar\x96#!2024-01-01#!75.0#!True\n")
    # Inject a truly invalid UTF-8 byte: 0x96
    f.write(b"7#!SecondInvalidChar\x96#!2024-01-01#!100.0#!True\n")
    # Inject a NUL byte (\x00) in the name field
    f.write(b"8#!D\x00vid#!2023-07-30#!88.5#!False\n")  # This will trigger _csv.Error

batches = CSVBatchReader(filepath="./data/sample.csv",
                        batch_size=2,
                        delimiter="#!",
                        encoding="utf-8",
                        drop_headers=True,
                        headers=['id', 'name', 'signup_date', 'score', 'active']
                        )

with batches as f:
    for i, batch in enumerate(f, start=1):
        print(f"Batch: {i}")
        for line in batch:
            print(line)

print(list(next(f)))

print("Done")

csv_reader.py:124 in _get_batch() at 00:56:48.291
csv_reader.py:55 in file() at 00:56:48.291
csv_reader.py:97 in resolve_headers() at 00:56:48.291
csv_reader.py:112 in _get_first_line() at 00:56:48.291
csv_reader.py:55 in file() at 00:56:48.291
csv_reader.py:117 in _get_first_line() at 00:56:48.291
raw_line: b'6#!InvalidChar\x96#!2024-01-01#!75.0#!True
          '
raw_line: b'7#!SecondInvalidChar\x96#!2024-01-01#!100.0#!True
          '
csv_reader.py:55 in file() at 00:56:48.294
csv_reader.py:97 in resolve_headers() at 00:56:48.294
csv_reader.py:112 in _get_first_line() at 00:56:48.294
csv_reader.py:55 in file() at 00:56:48.294
Batch: 1
{'id': '2', 'name': 'Bob', 'signup_date': '2023-03-22', 'score': '92.0', 'active': 'False'}
{'id': '3', 'name': 'Charlie', 'signup_date': '2023-05-10', 'score': '78.0', 'active': 'True'}
csv_reader.py:124 in _get_batch() at 00:56:48.295
csv_reader.py:55 in file() at 00:56:48.295
Batch: 2
{'id': '4', 'name': 'David', 'signup_date': '2023-07-30', 'score':

In [4]:
# Save as encoded_sample_cp1252.csv
data = [
    "1#!Alice#!2023-01-15#!85.5#!True\n",
    "2#!Bob#!2023-03-22#!92.0#!False\n",
    "3#!Charlie#!2023-05-10#!78.0#!True\n",
    "4#!David#!2023-07-30#!88.5#!False\n",
    "5#!Eva#!2023-10-01#!95.0#!True\n",
    "6#!InvalidChar\u2013#!2024-01-01#!75.0#!True\n",         # \u2013 is an en-dash (0x96 in cp1252)
    "7#!SecondInvalidChar\u2013#!2024-01-01#!100.0#!True\n",
    "8#!D\u0000vid#!2023-07-30#!88.5#!False\n"                # Includes a NUL character
]

with open("./data/sample_cp1252.csv", "w", encoding="cp1252", errors="strict") as f:
    f.writelines(data)

In [5]:
batches = CSVBatchReader(filepath="./data/sample_cp1252.csv",
                        batch_size=2,
                        delimiter="#!",
                        encoding="utf-8",
                        drop_headers=False,
                        headers =['id', 'name', 'signup_date', 'score', 'active']
                        )

with batches as f:
    next(f)
    # for i, batch in enumerate(f, start=1):
    #     print(f"Batch: {i}")
    #     for line in batch:
    #         print(line)

print("Done")

csv_reader.py:123 in _get_batch() at 00:49:20.628
csv_reader.py:55 in file() at 00:49:20.629
csv_reader.py:96 in resolve_headers() at 00:49:20.629
raw_line: b'6#!InvalidChar\x96#!2024-01-01#!75.0#!True
          '
raw_line: b'7#!SecondInvalidChar\x96#!2024-01-01#!100.0#!True
          '
csv_reader.py:55 in file() at 00:49:20.631
csv_reader.py:96 in resolve_headers() at 00:49:20.631
csv_reader.py:55 in file() at 00:49:20.631
Done


In [6]:
ic.disable()

In [7]:
with batches as f:
    next(f)
    raise ValueError('Custom exception to check exit method')

print("Done")

Done


In [8]:
import chardet

# Read the file as raw bytes
with open("./data/sample_cp1252.csv", "rb") as f:
    raw_data = f.read()

# Detect encoding
result = chardet.detect(raw_data)

# Print detected encoding and confidence
print(f"Detected encoding: {result['encoding']}")
print(f"Confidence: {result['confidence']}")

Detected encoding: Windows-1252
Confidence: 0.73


Assuming now that we want to apply a function before we pass the file into the DictReader. For example DictReader does not support double character delimiter but the received file uses #! as delimiter.

In [92]:
# Read the file into a string
with open("./data/sample_data.csv", "r", encoding="utf-8") as f:
    content = f.read()

# Replace delimiters
new_content = content.replace(",", "#!")

# Write the modified content to a new file
with open("./data/new_sample_data.csv", "w", encoding="utf-8", newline="") as f:
    f.write(new_content)


In [130]:

class CSVBatchReader:
    def __init__(self, filename, batch_size, delimiter=',', headers=None, hook=None):
        self.filename = filename
        self.batch_size = batch_size
        self.delimiter = delimiter
        self.raw_headers = headers
        self.headers = headers
        self.file = None
        self.hook = hook

    def __iter__(self):
        self.file = open(self.filename, newline='')
        if self.raw_headers is None:
            first_line = next(self.file).strip("\r\n")
            self.headers = first_line.split(self.delimiter)
        return self

    def __next__(self):

        batch_lines = list(islice(self.file, self.batch_size))

        # if batch_lines is an empty list close the file
        if not batch_lines:
            self.file.close()
            raise StopIteration


        return DictReader(batch_lines, delimiter=self.delimiter, fieldnames=self.headers)


    @validate_delimiter_change
    def replace_delimiter(self, lines, /, old_delimiter=None, new_delimiter=","):
        print('replace_delimiter was called ...')
        if old_delimiter is None:
            old_delimiter = self.delimiter
        return (line.replace(old_delimiter, new_delimiter) for line in lines)


In [131]:
batches = CSVBatchReader(filename="./data/new_sample_data.csv",
                         batch_size=2,
                         delimiter="#!",
                         hook=partial(validate_delimiter_change, old_delimiter="#!", new_delimiter=","))
for batch in batches:
    for line in batch:
        print(line)

functools.partial(<AdapterWrapper at 0x1048d3ed0 for function at 0x104b256c0>, old_delimiter='#!', new_delimiter=',') True
['1#!Alice#!2023-01-15#!85.5#!True\n', '2#!Bob#!2023-03-22#!92.0#!False\n']


TypeError: "delimiter" must be a 1-character string

In [111]:
from functools import reduce

l = ['this is the first line', 'this is the second line', 'this is the third line']

def custom_join(delimiter, items):
    if not items:
        return ''
    return reduce(lambda t_1, t_2: t_1 + delimiter + t_2 + "\n", items)

text = custom_join(",", l)

In [112]:
text

'this is the first line,this is the second line\n,this is the third line\n'

In [113]:
custom_join(", ", [])

''